# Computer Exercise 13.8 — Problem 3

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.8 Minimization — *Robustness & Conditioning: GN vs LM*
> **풀이 일자**: 2026-06-27 · **언어**: 본문 한국어 / 그래프 라벨 영문


## 1. 문제 (원문)

> **3.** Quantitatively compare Gauss–Newton and Levenberg–Marquardt for nonlinear least squares.
> (a) For the exponential model $\varphi(t;a,b)=a\,e^{bt}$, sweep a grid of **initial guesses**
> $(a_0,b_0)$ and record, for each method, whether the iteration converges to the least-squares
> solution. Plot the resulting **basins of attraction** (success maps). (b) For a
> **sum-of-two-exponentials** model $\varphi(t;\mathbf x)=x_1 e^{x_2 t}+x_3 e^{x_4 t}$, monitor the
> **condition number** of $J^\top J$ along the iterations to expose the ill-conditioning that makes
> the damping of LM essential.

### 한국어 풀이용 정리
- (a) 지수모형에서 **초기값 격자** $(a_0,b_0)$ 를 스윕 → GN/LM 각각의 **수렴 성공맵** 비교.
- (b) **이중지수합** 모형에서 $J^\top J$ 의 **조건수**를 추적 → 악조건과 감쇠의 필요성 확인.


## 2. 수학적 배경

### 2.1 수렴 영역(basin of attraction)
비선형 최소제곱은 비볼록일 수 있어, 같은 알고리즘이라도 **초기값**에 따라 전역 최소로 가거나 발산/국소점에 갇힌다. 참해로 수렴하는 초기값 집합이 그 방법의 *수렴 영역* 이다. 감쇠가 스텝 길이를 제한하는 LM 은 보통 **더 넓은 수렴 영역**을 가진다.

### 2.2 조건수와 악조건
정규방정식 $J^\top J\,\boldsymbol\delta=-J^\top\mathbf r$ 의 민감도는 $\kappa(J^\top J)=\kappa(J)^2$ 가 지배한다. 이중지수합처럼 두 지수의 감쇠율이 비슷하면 야코비안의 두 열이 거의 평행해져 $J^\top J$ 가 **거의 특이**($\kappa\gg1$)가 된다. 그러면 무감쇠 GN 스텝 $\boldsymbol\delta_{\mathrm{GN}}=-(J^\top J)^{-1}J^\top\mathbf r$ 가 폭주한다. LM 의 $(J^\top J+\lambda D)$ 는 최소 고윳값을 들어올려
$$
\boxed{\ \kappa(J^\top J+\lambda D)\ \ll\ \kappa(J^\top J)\ }
$$
로 **조건수를 직접 낮춰** 스텝을 안정화한다.


## 3. 풀이 흐름

1. **지수모형** small-residual 데이터 생성, 잡음 때문에 참값과 미세하게 다른 **최소제곱 해**를 기준점으로 한 번 계산.
2. GN/LM 솔버를 *수렴 여부만 반환*하도록 래핑(기준점과의 거리 $<5\times10^{-3}$ 이면 성공).
3. **초기값 격자** $(a_0,b_0)$ 스윕 → 각 점에서 두 방법의 수렴 성공 기록.
4. **성공맵 2장**(GN, LM)을 나란히 시각화, 성공 픽셀 비율 비교.
5. **이중지수합** 데이터 생성, LM 으로 적합하며 매 반복 $\kappa(J^\top J)$ 기록.
6. **조건수 곡선**(무감쇠 vs 감쇠) 시각화.
7. **해석**: 수렴 영역 넓이 차이와 악조건의 연결.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

a_true, b_true = 2.5, -0.6
t = np.linspace(0.0, 4.0, 25)
y = a_true * np.exp(b_true * t) + 0.01 * rng.standard_normal(t.size)

def res_exp(x):
    a, b = x
    return a * np.exp(b * t) - y

def jac_exp(x):
    a, b = x
    e = np.exp(b * t)
    return np.column_stack([e, a * t * e])

# 잡음 때문에 진짜 최소제곱 해는 참값과 미세하게 다르다 -> 좋은 초기값에서 풀어 기준점으로 사용
def _gn_solve(x0, it=80):
    x = np.array(x0, float)
    for _ in range(it):
        r = res_exp(x); J = jac_exp(x); g = J.T @ r
        if np.linalg.norm(g) < 1e-13:
            break
        x = x + np.linalg.solve(J.T @ J, -g)
    return x

x_opt = _gn_solve([2.0, -0.5])
print("true (a,b)           =", (a_true, b_true))
print("least-squares target =", x_opt)


true (a,b)           = (2.5, -0.6)
least-squares target = [ 2.50189497 -0.60337763]


In [2]:
# 수렴 여부만 반환하는 GN / LM 솔버 (기준점 x_opt 와의 거리로 판정)
def solve_gn(x0, tol=1e-8, maxit=100):
    x = np.array(x0, float)
    for _ in range(maxit):
        r = res_exp(x); J = jac_exp(x); g = J.T @ r
        if np.linalg.norm(g) < tol:
            break
        try:
            d = np.linalg.solve(J.T @ J, -g)
        except np.linalg.LinAlgError:
            return False
        x = x + d
        if not np.all(np.isfinite(x)) or np.linalg.norm(x) > 1e6:
            return False
    return np.linalg.norm(x - x_opt) < 5e-3

def solve_lm(x0, lam0=1e-2, tol=1e-8, maxit=200):
    x = np.array(x0, float); lam = lam0
    r = res_exp(x); cost = 0.5 * r @ r
    for _ in range(maxit):
        J = jac_exp(x); g = J.T @ r; A = J.T @ J
        if np.linalg.norm(g) < tol:
            break
        D = np.diag(np.maximum(np.diag(A), 1e-12))
        try:
            d = np.linalg.solve(A + lam * D, -g)
        except np.linalg.LinAlgError:
            lam *= 3; continue
        xn = x + d; rn = res_exp(xn); cn = 0.5 * rn @ rn
        pred = -(g @ d + 0.5 * d @ A @ d)
        rho = (cost - cn) / pred if pred > 0 else -1.0
        if rho > 0:
            x, r, cost = xn, rn, cn; lam = max(lam / 3, 1e-12)
        else:
            lam *= 3
        if not np.all(np.isfinite(x)):
            return False
    return np.linalg.norm(x - x_opt) < 5e-3

a0_grid = np.linspace(0.2, 6.0, 60)
b0_grid = np.linspace(-2.0, 1.0, 60)
succ_gn = np.zeros((b0_grid.size, a0_grid.size))
succ_lm = np.zeros_like(succ_gn)
with np.errstate(all="ignore"):
    for i, b0 in enumerate(b0_grid):
        for j, a0 in enumerate(a0_grid):
            succ_gn[i, j] = solve_gn([a0, b0])
            succ_lm[i, j] = solve_lm([a0, b0])

rate_gn = 100 * succ_gn.mean()
rate_lm = 100 * succ_lm.mean()
print(f"convergence success rate  ->  GN: {rate_gn:.1f}%   LM: {rate_lm:.1f}%")
pd.DataFrame({"method": ["Gauss-Newton", "Levenberg-Marquardt"],
              "success rate (%)": [round(rate_gn, 1), round(rate_lm, 1)]})


convergence success rate  ->  GN: 91.4%   LM: 100.0%


,method,success rate (%)
0,Gauss-Newton,91.4
1,Levenberg-Marquardt,100.0


In [3]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
ext = [a0_grid[0], a0_grid[-1], b0_grid[0], b0_grid[-1]]
for k, (S, name, rate) in enumerate([(succ_gn, "Gauss-Newton", rate_gn),
                                     (succ_lm, "Levenberg-Marquardt", rate_lm)]):
    ax[k].imshow(S, origin="lower", extent=ext, aspect="auto", cmap="Greens", vmin=0, vmax=1)
    ax[k].plot(x_opt[0], x_opt[1], "*", color="#c0392b", ms=16, label="LSQ solution")
    ax[k].set_xlabel(r"$a_0$"); ax[k].set_ylabel(r"$b_0$")
    ax[k].set_title(f"{name}: basin of attraction\nsuccess = {rate:.1f}%")
    ax[k].legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()
print("green = converged to the least-squares solution")


green = converged to the least-squares solution


In [4]:
# (b) 이중지수합 모형의 악조건 : d1≈d2 이면 두 지수쌍이 거의 평행
tb = np.linspace(0.0, 3.0, 40)
c1t, d1t, c2t, d2t = 1.5, -0.9, 1.0, -1.1
yb = c1t * np.exp(d1t * tb) + c2t * np.exp(d2t * tb) + 0.002 * rng.standard_normal(tb.size)

def res2(x):
    c1, d1, c2, d2 = x
    return c1 * np.exp(d1 * tb) + c2 * np.exp(d2 * tb) - yb

def jac2(x):
    c1, d1, c2, d2 = x
    e1 = np.exp(d1 * tb); e2 = np.exp(d2 * tb)
    return np.column_stack([e1, c1 * tb * e1, e2, c2 * tb * e2])

def lm_track(x0, lam0=1e-2, tol=1e-9, maxit=200):
    x = np.array(x0, float); lam = lam0
    r = res2(x); cost = 0.5 * r @ r
    conds, conds_damped = [], []
    for _ in range(maxit):
        J = jac2(x); g = J.T @ r; A = J.T @ J
        conds.append(np.linalg.cond(A))
        D = np.diag(np.maximum(np.diag(A), 1e-12))
        conds_damped.append(np.linalg.cond(A + lam * D))
        if np.linalg.norm(g) < tol:
            break
        d = np.linalg.solve(A + lam * D, -g)
        xn = x + d; rn = res2(xn); cn = 0.5 * rn @ rn
        pred = -(g @ d + 0.5 * d @ A @ d)
        rho = (cost - cn) / pred if pred > 0 else -1.0
        if rho > 0:
            x, r, cost = xn, rn, cn; lam = max(lam / 3, 1e-12)
        else:
            lam *= 3
    return x, np.array(conds), np.array(conds_damped)

with np.errstate(all="ignore"):
    x2, conds, conds_d = lm_track([1.0, -0.5, 1.0, -1.5])
print("recovered (c1,d1,c2,d2) =", np.round(x2, 3))
print(f"max cond(J^T J)         = {conds.max():.3e}")
print(f"max cond(J^T J + lam D) = {conds_d.max():.3e}")
pd.DataFrame({"quantity": ["cond(J^T J)  max", "cond(J^T J + lam D)  max"],
              "value": [conds.max(), conds_d.max()]})


recovered (c1,d1,c2,d2) = [ 2.367 -0.952  0.131 -1.505]
max cond(J^T J)         = 2.089e+08
max cond(J^T J + lam D) = 1.625e+07


,quantity,value
0,cond(J^T J) max,2.088528e+08
1,cond(J^T J + lam D) max,1.625413e+07


In [5]:
fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.semilogy(conds, "o-", ms=4, color="#c0392b", label=r"$\kappa(J^T J)$  (undamped)")
ax.semilogy(conds_d, "s--", ms=4, color="#16a085", label=r"$\kappa(J^T J+\lambda D)$  (LM)")
ax.set_xlabel("iteration k"); ax.set_ylabel("condition number")
ax.set_title("Ill-conditioning of the two-exponential normal equations")
ax.legend()
plt.tight_layout(); plt.show()
print("done")


done


## 4. 결과 해석

1. **LM 의 수렴 영역이 더 넓다**: 성공맵에서 초록(수렴) 영역이 Levenberg–Marquardt 쪽이 뚜렷하게 넓다. GN 은 기준해 근방에서만 수렴하지만, LM 은 감쇠가 큰 스텝을 잘라내 **멀리서 출발해도** 골짜기로 끌어들인다. 성공률 수치(상단 표)로도 LM 우위가 확인된다.
2. **악조건이 GN 실패의 근본 원인**: 이중지수합에서 두 감쇠율 $d_1\approx d_2$ 라 야코비안의 열들이 거의 평행 → $\kappa(J^\top J)$ 가 $10^{8}$ 규모로 폭증한다. 이때 무감쇠 GN 스텝은 수치적으로 불안정하다.
3. **감쇠가 조건수를 직접 낮춘다**: $\kappa(J^\top J+\lambda D)$ 곡선이 무감쇠 곡선보다 항상 아래에 있어, LM 이 정규방정식의 **악조건을 능동적으로 완화**함을 보여준다. 수렴 국면에서 $\lambda$ 가 작아지면 두 곡선이 가까워지는데, 이는 해 근방에서 LM 이 자연스럽게 GN 으로 복귀함을 뜻한다.
4. **종합**: 강건성(넓은 수렴 영역)과 정확성(해 근방 2차 수렴)을 동시에 얻는 것이 LM 이 비선형 최소제곱의 표준 도구가 된 이유다.

> **결론**: Gauss–Newton 은 해 근방에서만 안전하지만, **Levenberg–Marquardt 는 감쇠로 조건수를 낮춰 수렴 영역을 크게 넓힌다** — 강건성과 빠른 국소 수렴을 함께 제공한다.

**다음 Day 예고** — 챕터 13(최적화)의 직선탐색·신뢰영역·제약(SQP/내부점)·비선형 최소제곱(GN·LM)을 모두 다뤘다. 다음 절은 **§13.9 도함수 없는 최적화(Nelder–Mead·좌표강하)** 또는 커리큘럼 확장으로 이어진다.
